# samebit: Qwen2.5-0.5B in integers on an NVIDIA GPU

Settings → Accelerator → **GPU T4 x2** (or P100), Internet **on**. Then *Run all* (about 10 minutes).

The last cell prints a report: whether the GPU gives the same bits as the CPU, and the speed
against ordinary float16 / float32 inference.

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv

In [ ]:
!git clone -q https://github.com/Shakhtar-Sankur/samebit
%cd samebit
!pip install -q -e ".[test]"
try:
    import cupy
except ImportError:
    !pip install -q cupy-cuda12x
import cupy

print("cupy", cupy.__version__)

In [ ]:
# Qwen2.5-0.5B (pinned revision) and WikiText-2, about 1 GB
import os
import shutil

from huggingface_hub import hf_hub_download, snapshot_download

WORK = "/kaggle/working/qwen"
os.environ["QWEN_WORK"] = WORK
snapshot_download(
    "Qwen/Qwen2.5-0.5B",
    revision="060db6499f32faf8b98477b0a26969ef7d8b9987",
    local_dir=f"{WORK}/model",
)
os.makedirs(f"{WORK}/data", exist_ok=True)
for split in ("test", "train"):
    p = hf_hub_download(
        "Salesforce/wikitext",
        f"wikitext-2-raw-v1/{split}-00000-of-00001.parquet",
        repo_type="dataset",
    )
    shutil.copy(p, f"{WORK}/data/wt2-{split}.parquet")
print(sorted(os.listdir(f"{WORK}/model")))

In [ ]:
# Convert to samebit v0.2 integers (expected fingerprint e1994bc2e4f9bff7)
!cd tools/qwen2 && python convert.py 2>&1 | grep -v "^layer"

In [ ]:
# Every v0.2 op on the GPU must equal the NumPy op exactly
!python -m pytest -q tests/test_v2_cuda.py tests/test_v2_ops.py

In [ ]:
# Same bits GPU vs CPU, any summation order, and speed
!cd tools/qwen2 && python gpu_check.py 2>&1 | grep -v "^\\[transformers\\]"